In [0]:
# %sql
# drop schema pysparkdbt.gold cascade

In [0]:
df = spark.read.format('csv')\
.option('header', True)\
.option('inferSchema', True)\
  .load('/Volumes/pysparkdbt/source/source_data/customers/customers.csv')

In [0]:
display(df.limit(5))

In [0]:
df.schema

In [0]:
schema_customer = df.schema
schema_customer

### Spark Streaming

In [0]:
# df = spark.readStream.format("csv")\
# .option('header', True)\
# .schema(schema_customer)\
#   .load('/Volumes/pysparkdbt/source/source_data/customers/customers.csv')
  
# # checkpointLocation
# df.writeStream.format("delta")\
# .outputMode("append")\
# .option("checkpointLocation", "/Volumes/pysparkdbt/source/checkpoint/customers")\
#     .trigger(once=True)\
# .toTable("pysparkdbt.bronze.customers")

In [0]:
# array = [
#     {"name": "customers",
#     "schema": "StructType([StructField('customer_id', IntegerType(), True), StructField('first_name', StringType(), True), StructField('last_name', StringType(), True), StructField('email', StringType(), True), StructField('phone_number', StringType(), True), StructField('city', StringType(), True), StructField('signup_date', DateType(), True), StructField('last_updated_timestamp', TimestampType(), True)])"}
# ]

### dynamic solution: checkpoint volume bronze schema

In [0]:
entities = ['customers', 'payments', 'locations','trips','vehicles','drivers']

In [0]:
for entity in entities:

    df_batch = spark.read.format('csv')\
    .option('header', True)\
    .option('inferSchema', True)\
    .load(f"/Volumes/pysparkdbt/source/source_data/{entity}/")

    schema_entity = df_batch.schema
  
    df = spark.readStream.format("csv")\
        .option('header', True)\
        .schema(schema_entity)\
        .load(f"/Volumes/pysparkdbt/source/source_data/{entity}")

    df.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", f"/Volumes/pysparkdbt/source/checkpoint/{entity}")\
            .trigger(once=True)\
    .toTable(f"pysparkdbt.source.{entity}")  


In [0]:
%sql
select * from pysparkdbt.source.customers 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.customers 

In [0]:
%sql
select customer_id, count(*)
from pysparkdbt.source.customers
group by customer_id
having count(*) > 1

In [0]:
%sql
select * from pysparkdbt.source.drivers 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.drivers

In [0]:
%sql
select driver_id, count(*)
from pysparkdbt.source.drivers
group by driver_id
having count(*) > 1

In [0]:
%sql
select * from pysparkdbt.source.locations 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.locations

In [0]:
%sql
select location_id, count(*)
from pysparkdbt.source.locations
group by location_id
having count(*) > 1

In [0]:
%sql
select * from pysparkdbt.source.payments 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.payments

### test duplicate data

In [0]:
%sql
select payment_id, count(*)
from pysparkdbt.source.payments
group by payment_id
having count(*) > 1

In [0]:
%sql
select * from pysparkdbt.source.trips 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.trips

In [0]:
%sql
select trip_id, count(*)
from pysparkdbt.source.trips
group by trip_id
having count(*) > 1

In [0]:
%sql
select * from pysparkdbt.source.vehicles 
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.source.vehicles

In [0]:
%sql
select vehicle_id, count(*)
from pysparkdbt.source.vehicles
group by vehicle_id
having count(*) > 1

In [0]:
# %sql
# drop schema pysparkdbt.default_bronze cascade 

### Bronze layer

In [0]:
%sql
select * from pysparkdbt.bronze.stg_customers
limit 5

### Silver layer

In [0]:
%sql
select * from pysparkdbt.silver.silver_customers
limit 5

In [0]:
%sql
select * from pysparkdbt.silver.silver_drivers
limit 5

In [0]:
%sql
select count(*) from pysparkdbt.silver.silver_drivers

In [0]:
%sql
select count(*) from pysparkdbt.silver.silver_locations

In [0]:
%sql
select * 
from pysparkdbt.silver.silver_payments
limit 3

In [0]:
%sql
select count(*) from pysparkdbt.silver.silver_payments

In [0]:
%sql
select count(*) from pysparkdbt.silver.silver_vehicles

In [0]:
%sql
select count(*) from pysparkdbt.silver.silver_trips

In [0]:
%sql
select * from pysparkdbt.source.trips 
limit 5

In [0]:
%sql
describe table pysparkdbt.source.trips 

In [0]:
%sql
describe table pysparkdbt.source.payments 